![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 9 -- Lab 1: Judge the Admissions Model

Yesterday's homework built a logistic regression for the Hungarian AI Olympiad's admissions task and reported one number, accuracy. Today you judge that same model the way the organisers do: the majority baseline, the confusion matrix, precision, recall and F1, what moving the threshold does, ROC-AUC on probabilities against ROC-AUC on labels, and a cross-validated score. You end with the submission the task really asks for: probabilities, not labels.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** `train.csv` (6,885 students) and `test.csv` (765) from the HAIO 2025 repository, read from GitHub as in Day 08's Lab 3. The setup cell renames the columns and applies that lab's preparation (the `-1` flags and the one-hot county) in one function, so you start from the model-ready table `X`.

---
# Setup

In [ ]:
# pandas, matplotlib and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
BASE = "https://raw.githubusercontent.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad/main/2025/nyari-online/adatok/"
EN = {"Életkor": "age", "Nem": "sex", "Osztályzat_9": "grade_9", "Osztályzat_10": "grade_10", "Osztályzat_11": "grade_11",
      "Osztályzat_12": "grade_12", "Történelem": "history", "Matematika": "math", "Magyar Nyelv és Irodalom": "hungarian",
      "Informatika": "informatics", "Biológia": "biology", "Fizika": "physics", "Angol": "english", "Német": "german",
      "Informatika_emelt": "informatics_adv", "Biológia_emelt": "biology_adv", "Fizika_emelt": "physics_adv",
      "Angol_emelt": "english_adv", "Német_emelt": "german_adv", "Matematika_emelt": "math_adv", "Történelem_emelt": "history_adv",
      "Magyar Nyelv és Irodalom_emelt": "hungarian_adv", "Szülői Végzettség": "parent_education",
      "Középiskola Presztízse": "school_prestige", "Extrakurrikuláris Tevékenységek": "extracurricular",
      "Tanulási Szokások": "study_habits", "Munkatapasztalat": "work_experience", "Ajánlások Száma": "recommendations",
      "Versenyeken Való Részvétel": "competitions", "Vármegye": "county", "Felvételi Eredmény": "admitted", "ID": "ID"}
train = pd.read_csv(BASE + "train.csv").rename(columns=EN)
test = pd.read_csv(BASE + "test.csv").rename(columns=EN)

In [ ]:
def prepare(d):
    """Day 08 Lab 3 in one function: -1 becomes a flag plus a zero, the county becomes one-hot columns."""
    d = d.copy()
    for c in ["informatics", "biology", "physics", "english", "german"]:
        d[c + "_taken"] = (d[c] != -1).astype(int)
        d[c] = d[c].replace(-1, 0)
        d[c + "_adv"] = d[c + "_adv"].replace(-1, 0)
    feats = [c for c in d.columns if c not in ("county", "admitted", "ID")]
    return pd.concat([d[feats], pd.get_dummies(d["county"], prefix="county", dtype=int)], axis=1)

X = prepare(train)
y = train["admitted"].astype(int)
X_test = prepare(test).reindex(columns=X.columns, fill_value=0)
print(X.shape, X_test.shape)

---
# Part 1 -- The model and the number to beat

The *When Accuracy Lies* slide: before scoring anything, compute what a model that never looks at the data would score.

## Task 1: Split and majority baseline

Split with `train_test_split(X, y, test_size=0.2, random_state=42)` into `X_train, X_val, y_train, y_val`. The majority baseline on the validation rows is the share of the bigger class: compute `majority = max(y_val.mean(), 1 - y_val.mean())` and print it (expected 0.705).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(majority - 0.705) < 0.005, 'majority: share of the larger class among y_val'
print('Task 1 passed')

## Task 2: Fit, predict labels, predict probabilities

Fit a `StandardScaler` on `X_train`, then a `LogisticRegression(max_iter=1000)` on the scaled training rows. Store `pred = model.predict(...)` (0/1 labels) and `proba = model.predict_proba(...)[:, 1]` (probability of admission) for the scaled validation rows. Print the accuracy (expected 0.871).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(accuracy_score(y_val, pred) - 0.871) < 0.005, 'pred: labels from model.predict on the scaled validation rows'
assert proba.min() >= 0 and proba.max() <= 1 and len(set(np.round(proba, 2))) > 10, 'proba: column 1 of predict_proba, real probabilities'
print('Task 2 passed')

---
# Part 2 -- The confusion matrix and the scores it makes

The *Confusion Matrix* and *Precision, Recall, F1* slides. Positive means admitted.

## Task 3: The four counts

Compute `cm = confusion_matrix(y_val, pred)` and print it. Unpack it with `tn, fp, fn, tp = cm.ravel()` and print the four names with their counts. Expected: 907 true negatives, 64 false positives, 113 false negatives, 293 true positives.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert (tn, fp, fn, tp) == (907, 64, 113, 293), 'cm.ravel() gives TN, FP, FN, TP in that order'
print('Task 3 passed')

## Task 4: Precision, recall and F1 by hand

From the four counts compute `precision = tp / (tp + fp)`, `recall = tp / (tp + fn)` and `f1 = 2 * precision * recall / (precision + recall)`. Print them with 3 decimals. Expected 0.821, 0.722, 0.768. Also compute the accuracy from the counts and check it matches Task 2.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(precision - 0.821) < 0.002 and abs(recall - 0.722) < 0.002 and abs(f1 - 0.768) < 0.002, 'precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = harmonic mean'
print('Task 4 passed')

## Task 5: The same three with scikit-learn

Print `precision_score(y_val, pred)`, `recall_score(y_val, pred)` and `f1_score(y_val, pred)`. They must match Task 4 exactly; from now on you use these functions.

In [ ]:
# Your code here


## Task 6: Score the model that never looks

Build `lazy = np.zeros(len(y_val), dtype=int)` (predict 'not admitted' for everyone) and print its accuracy and its F1. Expected: accuracy 0.705, F1 0.0. Write in a comment which of the two numbers exposes it.

In [ ]:
# Your code here


---
# Part 3 -- The threshold and ROC-AUC

The *Threshold* slide: `predict` cuts the probability at 0.5. Nothing forces that cut.

## Task 7: Move the threshold

For each threshold `t` in `[0.3, 0.5, 0.7]`, build labels `(proba >= t).astype(int)` and print `t` with the precision, recall and F1. Expected at 0.3: precision 0.773, recall 0.872, F1 0.819; at 0.7: precision 0.870, recall 0.579, F1 0.695. Store the F1 at 0.3 in `f1_03`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(f1_03 - 0.819) < 0.003, 'f1_03: F1 of the labels (proba >= 0.3).astype(int)'
print('Task 7 passed')

## Task 8: Plot precision and recall against the threshold

For 19 thresholds `np.linspace(0.05, 0.95, 19)` compute precision and recall of `(proba >= t)` and draw both curves on one plot with a legend, a title and axis labels. Where do the curves cross?

In [ ]:
# Your code here


## Task 9: ROC-AUC with probabilities, then with labels

Compute `auc_proba = roc_auc_score(y_val, proba)` and `auc_labels = roc_auc_score(y_val, pred)`. Expected 0.954 and 0.828. Explain in a comment why the second one is so much lower although both come from the same model.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(auc_proba - 0.954) < 0.005 and abs(auc_labels - 0.828) < 0.01, 'roc_auc_score on proba versus on pred'
print('Task 9 passed')

---
# Part 4 -- Cross-validation and the real submission

The *k-Fold Cross-Validation* slide: a pipeline scales inside every fold, so nothing leaks.

## Task 10: Five folds of ROC-AUC

Build `pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` and run `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")`. Print the five scores, their mean and their standard deviation. Expected mean about 0.950 with a spread of about 0.003. Then run the same on `X[["school_prestige"]]` only, the organisers' baseline: expected mean about 0.558.

Syntax hint (the shape of the call, not the answer):

```python
scores = cross_val_score(estimator, X, y, cv=5, scoring="roc_auc")
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(scores.mean() - 0.950) < 0.005, 'scores: 5-fold cross_val_score with scoring="roc_auc" on the pipeline'
print('Task 10 passed')

## Task 11: Submit probabilities, not labels

Fit `pipe` on all of `X` and `y`, compute the admission probabilities of `X_test` with `predict_proba(...)[:, 1]`, and save `pd.DataFrame({"ID": test["ID"], "admitted": probabilities})` to `submission.csv` with `index=False`. Read it back and print `describe()` of the `admitted` column: values between 0 and 1, not just 0 and 1.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
chk = pd.read_csv('submission.csv')
assert chk.shape == (765, 2) and chk['admitted'].nunique() > 2, 'submission.csv: ID and admitted, 765 rows, admitted holds probabilities'
print('Task 11 passed')

---
## Task 12: Reflect

The lazy model has accuracy 0.705 and the real one 0.871: how much of the real model's accuracy is 'free'? At threshold 0.3 the F1 is higher than at 0.5: name one situation where the organisers would still prefer 0.7. Why does the organisers' choice of ROC-AUC make the threshold question disappear, and what does it force you to submit?

*Your answers here*

---
## Conclusion

- **The number to beat**: the majority baseline (0.705) comes before any model, and accuracy alone cannot tell a lazy model from a good one.
- **Confusion matrix and its scores**: you computed precision, recall and F1 by hand from four counts and matched scikit-learn.
- **Threshold and ROC-AUC**: every F1 lives at one threshold; ROC-AUC scores the probabilities, and labels cap it at 0.828 instead of 0.954.
- **Cross-validation and submission**: five folds gave a mean and a spread, and the file you hand in holds probabilities, as the metric demands.

---

Made By **Sattam Altwaim**